# Run the experiment on a Kaggle T4

This notebook runs the whole pipeline on a GPU and produces every number in
`results/`. Everything else in this repo runs on a laptop; only the extraction
stage needs CUDA.

**Before you start**

1. Settings → Accelerator → **GPU T4 x2** (one is enough; NF4 keeps the 7B under 16 GB).
2. Settings → Internet → **On** (needed to fetch the model and the dataset).
3. Get this repo into the session — either set `REPO_URL` below to your GitHub
   remote, or upload the repo as a Kaggle Dataset and set `INPUT_DIR`.

Expected wall clock at `n_examples: 3000`: roughly 40–70 minutes for the
extraction, then a couple of minutes for everything else.

In [ ]:
# Point at the repo. Set exactly one of these.
REPO_URL = ""                       # e.g. "https://github.com/<user>/controlplane-cascade.git"
INPUT_DIR = "/kaggle/input/controlplane-cascade"   # used when REPO_URL is empty

import os, shutil, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working/controlplane")
if REPO_URL:
    if WORK.exists():
        shutil.rmtree(WORK)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORK)], check=True)
else:
    if WORK.exists():
        shutil.rmtree(WORK)
    shutil.copytree(INPUT_DIR, WORK)

os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("working directory:", Path.cwd())
print(sorted(p.name for p in Path.cwd().iterdir()))

In [ ]:
!pip install -q -r requirements.txt

## Stage 2 gate — does the model load, and does it answer?

TASKS.md Stage 2 asks for three things before the expensive stage: the model loads inside
the memory budget, one generated answer looks sane, and the resolved layer indices are
printed against the model's actual depth.

In [ ]:
import torch

from src.config import load_config, set_seeds, setup_logging
from src.model import describe_model, load_model_and_tokenizer, peak_memory_gb, sanity_generate

setup_logging()
config = load_config("config.yaml")
set_seeds(config.seed)

print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("config hash:", config.config_hash)

model, tokenizer = load_model_and_tokenizer(config)
info = describe_model(model, tokenizer, config)

print()
print("num_hidden_layers :", info["num_hidden_layers"])
print("hidden_size       :", info["hidden_size"])
print("probe layers      :", info["probe_layers"])
print("layer fractions   :", info["layer_fractions"])
print("padding side      :", info["padding_side"])
print("peak GPU memory   : %.2f GB" % peak_memory_gb())
print()
print("prompt template:")
print(info["example_prompt"])

In [ ]:
for question in [
    "Who wrote the novel 'Nineteen Eighty-Four'?",
    "What is the capital of Australia?",
    "Which element has the chemical symbol 'Au'?",
]:
    print(f"Q: {question}")
    print(f"A: {sanity_generate(model, tokenizer, question, config)!r}")
    print()

## Stage 3 pre-flight — do not skip this

Three checks before the GPU hour, per TASKS.md Stage 3:

1. the **left-padding equivalence check** on a batch of 4,
2. an **n=20 smoke run** whose completions you read by eye,
3. a **base-rate check** on those 20 — roughly half should be correct.

`--dry-run` writes nothing, so a bad result here costs a minute rather than an hour.
Free the notebook's model first so the subprocess gets the whole GPU.

In [ ]:
import gc

del model
gc.collect()
torch.cuda.empty_cache()

!python scripts/01_extract.py --config config.yaml --limit 20 --dry-run

**Read the output above before continuing.**

- Max deviation should be around `1e-3` or smaller. If the check failed it raised, and
  the padding is wrong — stop, do not work around it.
- The completions should be short answers, not echoes of the prompt or empty strings.
- Roughly half should be marked `OK`. `0/20` or `20/20` means the prompt or the matching
  rule is broken, not that the model is unusually bad or good.

## The full run

Extraction, probe, economics, latency, report. Any stage can be re-run alone afterwards
with `--from`, so a failure at stage 04 does not cost the extraction again.

In [ ]:
!python scripts/run_all.py --config config.yaml

## The result

In [ ]:
from IPython.display import Markdown, display

display(Markdown(Path("results/RESULTS.md").read_text(encoding="utf-8")))

## Take the artifacts home

Everything a reviewer needs is small — the JSON files, the two plots, `RESULTS.md` and the
rendered `README.md`. The activations (~150 MB) and the parquet files stay behind; they are
regenerable and are gitignored.

Download `results_bundle.zip` from the Kaggle output pane, unzip it over `results/` in your
local checkout, then commit with an `exp:` message recording the numbers that moved.

In [ ]:
import zipfile

bundle = Path("/kaggle/working/results_bundle.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for pattern in ("results/*.json", "results/*.png", "results/RESULTS.md", "README.md"):
        for path in Path(".").glob(pattern):
            zf.write(path, path)
            print("added", path)
print()
print("wrote", bundle, f"({bundle.stat().st_size / 1024:.1f} KB)")